# 1. Carga, exploración inicial y unión de datos

En este cuaderno realizo los primeros pasos del proyecto:
- Cargar los dos conjuntos de datos: las llamadas de campaña (`bank-additional.csv`) y los datos de clientes (`customer-details.xlsx`).
- Unificar las 3 hojas del archivo Excel.
- Cruzar ambas tablas mediante el identificador de cliente.
- Hacer una primera revisión del tamaño de los datos, tipos de columnas y valores nulos.


In [1]:
import os
import pandas as pd
import numpy as np

# Ajusto para poder ver todas las columnas
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)


### 1.1. Carga de los datos de campaña (`bank-additional.csv`)
Cargamos el archivo CSV y revisamos las primeras filas y dimensiones.


In [2]:
csv_path = '../data/raw/bank-additional.csv'
df_campaign = pd.read_csv(csv_path)

print(f"Filas y columnas de df_campaign: {df_campaign.shape}")
df_campaign.head()


Filas y columnas de df_campaign: (43000, 24)


,Unnamed: 0,age,job,marital,education,default,housing,loan,contact,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,date,latitude,longitude,id_
0,0,NaN,housemaid,MARRIED,basic.4y,0.000,0.000,0.000,telephone,261,1,999,0,NONEXISTENT,1.100,"93,994","-36,4","4,857",5191,no,2-agosto-2019,41.495,-71.233,089b39d8-e4d0-461b-87d4-814d71e0e079
1,1,57.000,services,MARRIED,high.school,NaN,0.000,0.000,telephone,149,1,999,0,NONEXISTENT,1.100,"93,994","-36,4",NaN,5191,no,14-septiembre-2016,34.601,-83.923,e9d37224-cb6f-4942-98d7-46672963d097
2,2,37.000,services,MARRIED,high.school,0.000,1.000,0.000,telephone,226,1,999,0,NONEXISTENT,1.100,"93,994","-36,4","4,857",5191,no,15-febrero-2019,34.939,-94.847,3f9f49b5-e410-4948-bf6e-f9244f04918b
3,3,40.000,admin.,MARRIED,basic.6y,0.000,0.000,0.000,telephone,151,1,999,0,NONEXISTENT,1.100,"93,994","-36,4",NaN,5191,no,29-noviembre-2015,49.041,-70.308,9991fafb-4447-451a-8be2-b0df6098d13e
4,4,56.000,services,MARRIED,high.school,0.000,0.000,1.000,telephone,307,1,999,0,NONEXISTENT,1.100,"93,994","-36,4",NaN,5191,no,29-enero-2017,38.033,-104.463,eca60b76-70b6-4077-80ba-bc52e8ebb0eb


### 1.2. Tipos de datos e información general
Revisamos las columnas, tipos de datos y los valores no nulos que detecta Pandas de entrada.


In [3]:
df_campaign.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43000 entries, 0 to 42999
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Unnamed: 0      43000 non-null  int64  
 1   age             37880 non-null  float64
 2   job             42655 non-null  object 
 3   marital         42915 non-null  object 
 4   education       41193 non-null  object 
 5   default         34019 non-null  float64
 6   housing         41974 non-null  float64
 7   loan            41974 non-null  float64
 8   contact         43000 non-null  object 
 9   duration        43000 non-null  int64  
 10  campaign        43000 non-null  int64  
 11  pdays           43000 non-null  int64  
 12  previous        43000 non-null  int64  
 13  poutcome        43000 non-null  object 
 14  emp.var.rate    43000 non-null  float64
 15  cons.price.idx  42529 non-null  object 
 16  cons.conf.idx   43000 non-null  object 
 17  euribor3m       33744 non-null 

### 1.3. Carga del archivo Excel (`customer-details.xlsx`)
El Excel contiene tres pestañas (`2012`, `2013`, `2014`).
Leemos cada hoja con `pd.read_excel()` y las juntamos en un solo DataFrame con `pd.concat()`.


In [4]:
excel_path = '../data/raw/customer-details.xlsx'
xl = pd.ExcelFile(excel_path)
print("Hojas del Excel:", xl.sheet_names)

sheets_data = []
for sheet in xl.sheet_names:
    temp_df = pd.read_excel(xl, sheet_name=sheet)
    temp_df['signup_cohort'] = sheet
    sheets_data.append(temp_df)

df_customers = pd.concat(sheets_data, ignore_index=True)
print(f"Filas y columnas de df_customers: {df_customers.shape}")
df_customers.head()


Hojas del Excel: ['2012', '2013', '2014']
Filas y columnas de df_customers: (43170, 8)


,Unnamed: 0,Income,Kidhome,Teenhome,Dt_Customer,NumWebVisitsMonth,ID,signup_cohort
0,0,161770,1,0,2012-04-04,29,089b39d8-e4d0-461b-87d4-814d71e0e079,2012
1,1,85477,1,1,2012-12-30,7,e9d37224-cb6f-4942-98d7-46672963d097,2012
2,2,147233,1,1,2012-02-02,5,3f9f49b5-e410-4948-bf6e-f9244f04918b,2012
3,3,121393,1,2,2012-12-21,29,9991fafb-4447-451a-8be2-b0df6098d13e,2012
4,4,63164,1,2,2012-06-20,20,eca60b76-70b6-4077-80ba-bc52e8ebb0eb,2012


### 1.4. Comprobación de claves y cruce de tablas
Comprobamos si los identificadores (`id_` en campaña e `ID` en clientes) son únicos antes de hacer la unión.


In [5]:
print("IDs únicos en campaña:", df_campaign['id_'].nunique(), "de", len(df_campaign))
print("IDs únicos en clientes:", df_customers['ID'].nunique(), "de", len(df_customers))

campaign_ids = set(df_campaign['id_'])
customer_ids = set(df_customers['ID'])

print(f"Clientes coincidentes en ambas tablas: {len(campaign_ids.intersection(customer_ids))}")
print(f"Clientes del Excel sin llamadas registradas: {len(customer_ids - campaign_ids)}")


IDs únicos en campaña: 43000 de 43000
IDs únicos en clientes: 43170 de 43170
Clientes coincidentes en ambas tablas: 43000
Clientes del Excel sin llamadas registradas: 170


### 1.5. Merge de ambos DataFrames
Hacemos un `inner join` por el identificador y eliminamos las columnas sobrantes (`Unnamed: 0` y el `ID` duplicado).


In [6]:
df_merged = pd.merge(
    df_campaign,
    df_customers,
    left_on='id_',
    right_on='ID',
    how='inner'
)

# Quito columnas de índices previos y el ID duplicado
cols_to_drop = [c for c in df_merged.columns if 'Unnamed' in c] + ['ID']
df_merged = df_merged.drop(columns=cols_to_drop)

print(f"Dimensiones tras el merge: {df_merged.shape}")
df_merged.head(3)


Dimensiones tras el merge: (43000, 29)


,age,job,marital,education,default,housing,loan,contact,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,date,latitude,longitude,id_,Income,Kidhome,Teenhome,Dt_Customer,NumWebVisitsMonth,signup_cohort
0,NaN,housemaid,MARRIED,basic.4y,0.000,0.000,0.000,telephone,261,1,999,0,NONEXISTENT,1.100,"93,994","-36,4","4,857",5191,no,2-agosto-2019,41.495,-71.233,089b39d8-e4d0-461b-87d4-814d71e0e079,161770,1,0,2012-04-04,29,2012
1,57.000,services,MARRIED,high.school,NaN,0.000,0.000,telephone,149,1,999,0,NONEXISTENT,1.100,"93,994","-36,4",NaN,5191,no,14-septiembre-2016,34.601,-83.923,e9d37224-cb6f-4942-98d7-46672963d097,85477,1,1,2012-12-30,7,2012
2,37.000,services,MARRIED,high.school,0.000,1.000,0.000,telephone,226,1,999,0,NONEXISTENT,1.100,"93,994","-36,4","4,857",5191,no,15-febrero-2019,34.939,-94.847,3f9f49b5-e410-4948-bf6e-f9244f04918b,147233,1,1,2012-02-02,5,2012


### 1.6. Detección inicial de nulos
Calculamos el número y porcentaje de valores nulos por columna para saber qué habrá que limpiar en el siguiente paso.


In [7]:
null_summary = pd.DataFrame({
    'nulos': df_merged.isna().sum(),
    'porcentaje_%': (df_merged.isna().sum() / len(df_merged)) * 100
})

null_summary = null_summary[null_summary['nulos'] > 0].sort_values(by='nulos', ascending=False)
null_summary


,nulos,porcentaje_%
euribor3m,9256,21.526
default,8981,20.886
age,5120,11.907
education,1807,4.202
loan,1026,2.386
housing,1026,2.386
cons.price.idx,471,1.095
job,345,0.802
date,248,0.577
marital,85,0.198


### 1.7. Distribución de la variable objetivo (`y`)
Vemos cuántos clientes han contratado el depósito (`yes`) y cuántos no (`no`).


In [8]:
print("Recuento de la variable objetivo:")
print(df_merged['y'].value_counts())

print("\nPorcentaje:")
print(df_merged['y'].value_counts(normalize=True) * 100)


Recuento de la variable objetivo:
y
no     38156
yes     4844
Name: count, dtype: int64

Porcentaje:
y
no    88.735
yes   11.265
Name: proportion, dtype: float64
